In [1]:
from skimage import feature
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import os
from pathlib import Path
import sys
from sklearn.model_selection import train_test_split, cross_val_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import accuracy_score, classification_report, average_precision_score
from sklearn.metrics import precision_recall_curve

from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier as DTC
from sklearn.ensemble import RandomForestClassifier as RFC
from sklearn.neighbors import KNeighborsClassifier as KNN
import xgboost as XGB


In [2]:
parent_folder = Path().resolve().parent
src_path = parent_folder / 'src'
sys.path.append(str(src_path))

from tools import get_embedding_birdnet

#env to use: clef

In [3]:
root_folder='../data/train_data/embedding/birdnet/'

In [4]:
df_pos = get_embedding_birdnet(root_folder, 1)
df_neg = get_embedding_birdnet(root_folder, 0)

In [5]:
df_neg["filename"] = df_neg["embed_name"].str.split("_").str[0]
df_pos["filename"] = df_pos["embed_name"].str.split("-").str[0].str[:-1]

In [6]:
df_pos['target'] = 1
df_neg['target'] = 0

In [7]:
df_neg.sample(5)

,embed_name,embedding,filename,target
604,FKlxjH_1516.birdnet.embeddings.txt,"[0.0, 0.04240163, 0.058650542, 0.28684658, 0.0...",FKlxjH,0
1446,oOw8he_1037.birdnet.embeddings.txt,"[0.0, 0.0, 0.16756462, 0.3193762, 0.12584898, ...",oOw8he,0
999,IlaTbo_1925.birdnet.embeddings.txt,"[0.0, 0.08084688, 1.8380672, 0.00095585984, 0....",IlaTbo,0
167,3dBhE6_888.birdnet.embeddings.txt,"[0.0, 0.19986872, 0.16232726, 0.39405155, 0.17...",3dBhE6,0
1120,jlcfzd_1372.birdnet.embeddings.txt,"[0.09878841, 0.15158181, 0.21108373, 0.7680955...",jlcfzd,0


Perform 5-fold split for the negative data only

In [8]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Create a new column to store fold numbers
df_neg["fold"] = -1  # Initialize with -1

for fold, (train_idx, test_idx) in enumerate(kf.split(df_neg)):
    df_neg.loc[test_idx, "fold"] = fold  # Assign fold number to test samples

<IPython.core.display.Javascript object>

In [9]:
fold = 0 # Run the rest of the code for each fold

In [10]:
df_neg_fold = df_neg[df_neg.fold==fold]
df_neg_fold = df_neg_fold.drop("fold", axis=1)

In [11]:
from sklearn.model_selection import GroupKFold, cross_val_score

# Combine and shuffle
df_combined = pd.concat([df_pos, df_neg_fold], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=22).reset_index(drop=True)

# Features, labels, groups
X = np.vstack(df_combined["embedding"].values)
y = df_combined["target"].values
groups = df_combined["filename"].values

# 5-fold grouped CV
cv = GroupKFold(n_splits=5)

SVM

In [60]:
from sklearn.svm import SVC

model = SVC(kernel="rbf", cache_size=500)

accuracies = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="accuracy"
)

print(f"Accuracy: {accuracies.mean():.4f} ± {accuracies.std(ddof=1):.4f}")

ap = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="average_precision"
)


print(f"AP: {ap.mean():.4f} ± {ap.std(ddof=1):.4f}")

Accuracy: 0.9810 ± 0.0089
AP: 0.9988 ± 0.0011


Random Forest

In [61]:
model = RFC(n_jobs = -1)

accuracies = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="accuracy"
)

print(f"Accuracy: {accuracies.mean():.4f} ± {accuracies.std(ddof=1):.4f}")

ap = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="average_precision"
)


print(f"AP: {ap.mean():.4f} ± {ap.std(ddof=1):.4f}")

Accuracy: 0.9680 ± 0.0168
AP: 0.9961 ± 0.0021


XGBoost

In [62]:
model = XGB.XGBClassifier(objective='binary:logistic')

accuracies = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="accuracy"
)

print(f"Accuracy: {accuracies.mean():.4f} ± {accuracies.std(ddof=1):.4f}")

ap = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="average_precision"
)


print(f"AP: {ap.mean():.4f} ± {ap.std(ddof=1):.4f}")

Accuracy: 0.9501 ± 0.0255
AP: 0.9893 ± 0.0100


In [12]:
import os
import joblib
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, average_precision_score
from sklearn.metrics import confusion_matrix

cv = GroupKFold(n_splits=5)

dataset = "ds1"

accuracies = []
aps = []
confusion_matrices = []

os.makedirs("saved_models", exist_ok=True)

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = SVC(kernel="rbf", cache_size=500)

    model.fit(X_train, y_train)

    # Save model
    joblib.dump(model, f"saved_models/{dataset}_svm_fold_{fold}.joblib")

    # Evaluate
    y_pred = model.predict(X_test)
    y_score = model.decision_function(X_test)

    accuracies.append(accuracy_score(y_test, y_pred))
    aps.append(average_precision_score(y_test, y_score))
    cm = confusion_matrix(y_test, y_pred)
    confusion_matrices.append(cm)

    np.savetxt(
        f"saved_models/confusion_matrix_fold_{fold}.csv",
        cm,
        fmt="%d",
        delimiter=","
    )

print("Accuracy per fold:", np.round(accuracies, 4))
print(f"Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies, ddof=1):.4f}")

print("AP per fold:", np.round(aps, 4))
print(f"AP: {np.mean(aps):.4f} ± {np.std(aps, ddof=1):.4f}")


Accuracy per fold: [0.9801 0.9801 0.97   0.98   0.995 ]
Accuracy: 0.9810 ± 0.0089
AP per fold: [0.9993 0.9997 0.9973 0.9979 0.9999]
AP: 0.9988 ± 0.0011


In [60]:
#plt.plot(recall, precision, marker='.')
#plt.xlabel('Recall')
#plt.ylabel('Precision')
#plt.title('Precision-Recall Curve')
#plt.show()